In [1]:
from pyspark.sql import SparkSession

spark=(SparkSession.builder.appName("Basic_spark").master("local[*]").getOrCreate())

In [2]:
spark

In [112]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","Male","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29"," ","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

In [7]:
emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"

In [113]:
emp=spark.createDataFrame(data=emp_data,schema=emp_schema)

In [9]:
emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [30]:
emp_final_1=emp.where("salary>50000")

In [33]:
emp_final.write.format("csv").save("data/output/1/emp.csv")

In [35]:
emp_final_1.rdd.getNumPartitions()

4

In [37]:
emp.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: string (nullable = true)



In [44]:
from pyspark.sql.functions import expr,col

emp_2=emp.select(col('salary').cast('int'))


In [45]:
emp_2.printSchema()

root
 |-- salary: integer (nullable = true)



In [48]:
from pyspark.sql.types import *

schema_1=StructType([
         StructField("name",StringType(),True),
         StructField("age",IntegerType(),True)
])

In [50]:
emp_3=emp.selectExpr("employee_id as emp_id","name","cast(age as int) as age","salary")

In [51]:
emp_3.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: string (nullable = true)



In [53]:
emp_4=emp_3.select("emp_id","name","age","salary").where("age>30")

In [54]:
emp_4.show()

+------+-------------+---+------+
|emp_id|         name|age|salary|
+------+-------------+---+------+
|   003|    Bob Brown| 35| 55000|
|   005|    Jack Chan| 40| 60000|
|   006|    Jill Wong| 32| 52000|
|   007|James Johnson| 42| 70000|
|   009|      Tom Tan| 33| 58000|
|   011|   David Park| 38| 65000|
|   012|   Susan Chen| 31| 54000|
|   013|    Brian Kim| 45| 75000|
|   015|  Michael Lee| 37| 63000|
|   017|  George Wang| 34| 57000|
|   019|  Steven Chen| 36| 62000|
|   020|    Grace Kim| 32| 53000|
+------+-------------+---+------+



In [55]:
emp_4.write.format("csv").save("data/output/2/emp.csv")

In [61]:
from pyspark.sql.types import _parse_datatype_string

schema_str="emp_id string,age int"

schema_spark=_parse_datatype_string(schema_str)


In [62]:
schema_spark

StructType([StructField('emp_id', StringType(), True), StructField('age', IntegerType(), True)])

In [63]:
# Casting Column
# select employee_id, name, age, cast(salary as double) as salary from emp
emp_casted=emp.select("employee_id","name","age",col("salary").cast("double"))

In [64]:
emp_casted.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: double (nullable = true)



In [65]:
emp_taxed=emp_casted.withColumn("tax",col("salary")*0.2)

In [68]:
# Literals
# select employee_id, name, age, salary, tax, 1 as columnOne, 'two' as columnTwo from emp_taxed
from pyspark.sql.functions import lit
emp_new_columns=emp_taxed.withColumn("columnOne",lit(1)).withColumn("columntwo",lit("two"))

In [69]:
emp_new_columns.show()

+-----------+-------------+---+-------+-------+---------+---------+
|employee_id|         name|age| salary|    tax|columnOne|columntwo|
+-----------+-------------+---+-------+-------+---------+---------+
|        001|     John Doe| 30|50000.0|10000.0|        1|      two|
|        002|   Jane Smith| 25|45000.0| 9000.0|        1|      two|
|        003|    Bob Brown| 35|55000.0|11000.0|        1|      two|
|        004|    Alice Lee| 28|48000.0| 9600.0|        1|      two|
|        005|    Jack Chan| 40|60000.0|12000.0|        1|      two|
|        006|    Jill Wong| 32|52000.0|10400.0|        1|      two|
|        007|James Johnson| 42|70000.0|14000.0|        1|      two|
|        008|     Kate Kim| 29|51000.0|10200.0|        1|      two|
|        009|      Tom Tan| 33|58000.0|11600.0|        1|      two|
|        010|     Lisa Lee| 27|47000.0| 9400.0|        1|      two|
|        011|   David Park| 38|65000.0|13000.0|        1|      two|
|        012|   Susan Chen| 31|54000.0|10800.0| 

In [72]:
emp_1=emp_new_columns.withColumnRenamed("employee_id","emp_id")

In [73]:
emp_1.show()

+------+-------------+---+-------+-------+---------+---------+
|emp_id|         name|age| salary|    tax|columnOne|columntwo|
+------+-------------+---+-------+-------+---------+---------+
|   001|     John Doe| 30|50000.0|10000.0|        1|      two|
|   002|   Jane Smith| 25|45000.0| 9000.0|        1|      two|
|   003|    Bob Brown| 35|55000.0|11000.0|        1|      two|
|   004|    Alice Lee| 28|48000.0| 9600.0|        1|      two|
|   005|    Jack Chan| 40|60000.0|12000.0|        1|      two|
|   006|    Jill Wong| 32|52000.0|10400.0|        1|      two|
|   007|James Johnson| 42|70000.0|14000.0|        1|      two|
|   008|     Kate Kim| 29|51000.0|10200.0|        1|      two|
|   009|      Tom Tan| 33|58000.0|11600.0|        1|      two|
|   010|     Lisa Lee| 27|47000.0| 9400.0|        1|      two|
|   011|   David Park| 38|65000.0|13000.0|        1|      two|
|   012|   Susan Chen| 31|54000.0|10800.0|        1|      two|
|   013|    Brian Kim| 45|75000.0|15000.0|        1|   

In [74]:
# column name with space

emp_2=emp_1.withColumnRenamed("columnTwo","column Two")

In [75]:
emp_2.printSchema()

root
 |-- emp_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- tax: double (nullable = true)
 |-- columnOne: integer (nullable = false)
 |-- column Two: string (nullable = false)



In [76]:
emp_dropped=emp_new_columns.drop("columnTwo","columnOne")


In [77]:
emp_dropped.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- tax: double (nullable = true)



In [78]:
emp_filtered=emp_dropped.where("tax>10000")

In [79]:
emp_filtered.show()

+-----------+-------------+---+-------+-------+
|employee_id|         name|age| salary|    tax|
+-----------+-------------+---+-------+-------+
|        003|    Bob Brown| 35|55000.0|11000.0|
|        005|    Jack Chan| 40|60000.0|12000.0|
|        006|    Jill Wong| 32|52000.0|10400.0|
|        007|James Johnson| 42|70000.0|14000.0|
|        008|     Kate Kim| 29|51000.0|10200.0|
|        009|      Tom Tan| 33|58000.0|11600.0|
|        011|   David Park| 38|65000.0|13000.0|
|        012|   Susan Chen| 31|54000.0|10800.0|
|        013|    Brian Kim| 45|75000.0|15000.0|
|        015|  Michael Lee| 37|63000.0|12600.0|
|        017|  George Wang| 34|57000.0|11400.0|
|        019|  Steven Chen| 36|62000.0|12400.0|
|        020|    Grace Kim| 32|53000.0|10600.0|
+-----------+-------------+---+-------+-------+



In [80]:
emp_limit=emp_filtered.limit(5)

In [81]:
emp_limit.show()

+-----------+-------------+---+-------+-------+
|employee_id|         name|age| salary|    tax|
+-----------+-------------+---+-------+-------+
|        003|    Bob Brown| 35|55000.0|11000.0|
|        005|    Jack Chan| 40|60000.0|12000.0|
|        006|    Jill Wong| 32|52000.0|10400.0|
|        007|James Johnson| 42|70000.0|14000.0|
|        008|     Kate Kim| 29|51000.0|10200.0|
+-----------+-------------+---+-------+-------+



In [83]:
emp_limit.show(2)

+-----------+---------+---+-------+-------+
|employee_id|     name|age| salary|    tax|
+-----------+---------+---+-------+-------+
|        003|Bob Brown| 35|55000.0|11000.0|
|        005|Jack Chan| 40|60000.0|12000.0|
+-----------+---------+---+-------+-------+
only showing top 2 rows



In [85]:
columns={
    "Tax":col("salary")*0.2,
    "ColumnOne":lit(1),
    "columnTwo":lit("two")
    
}

emp_final=emp.withColumns(columns)

In [86]:
emp_final.show()

+-----------+-------------+-------------+---+------+------+----------+-------+---------+---------+
|employee_id|department_id|         name|age|gender|salary| hire_date|    Tax|ColumnOne|columnTwo|
+-----------+-------------+-------------+---+------+------+----------+-------+---------+---------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|10000.0|        1|      two|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15| 9000.0|        1|      two|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|11000.0|        1|      two|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30| 9600.0|        1|      two|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|12000.0|        1|      two|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|10400.0|        1|      two|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|14000.0|        1|      two|
|        0

In [114]:
from pyspark.sql.functions import when
emp_gender_fixed=emp.withColumn("New_gender",when(col("gender")=="Male","M").when(col("gender")=="Female","F").otherwise(None))

In [95]:
emp_gender_fixed_1=emp.withColumn("new_column",expr("case when gender ='Male' then 'M' when gender='Female' then 'F' else null end"))

In [115]:
emp_gender_fixed_1.show()

+-----------+-------------+-------------+---+------+------+----------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_column|
+-----------+-------------+-------------+---+------+------+----------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|         F|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|         M|
|        010|   

In [98]:
from pyspark.sql.functions import regexp_replace

emp_name_fixed=emp_gender_fixed.withColumn("New_name",regexp_replace(col('Name'),'j','z'))

In [100]:
from pyspark.sql.functions import to_date
emp_date_fix=emp_name_fixed.withColumn("hire_date",to_date(col("hire_date"),'yyyy-MM-dd'))

In [102]:
emp_date_fix.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- New_gender: string (nullable = true)
 |-- New_name: string (nullable = true)



In [104]:
from pyspark.sql.functions import current_date,current_timestamp
emp_dated=emp_date_fix.withColumn("current_date",current_date()).withColumn("Time_stamp",current_timestamp())

In [106]:
emp_dated.show(truncate=False)

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|employee_id|department_id|name         |age|gender|salary|hire_date |New_gender|New_name     |current_date|Time_stamp                |
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|001        |101          |John Doe     |30 |Male  |50000 |2015-01-01|M         |John Doe     |2026-03-19  |2026-03-19 14:17:35.099107|
|002        |101          |Jane Smith   |25 |Female|45000 |2016-02-15|F         |Jane Smith   |2026-03-19  |2026-03-19 14:17:35.099107|
|003        |102          |Bob Brown    |35 |Male  |55000 |2014-05-01|M         |Bob Brown    |2026-03-19  |2026-03-19 14:17:35.099107|
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|F         |Alice Lee    |2026-03-19  |2026-03-19 14:17:35.099107|
|005        |103          |Jack Chan    |40 |Mal

In [110]:
emp_1=emp_dated.na.drop()

In [111]:
emp_1.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|New_gender|     New_name|current_date|          Time_stamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     John Doe|  2026-03-19|2026-03-19 14:19:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Jane Smith|  2026-03-19|2026-03-19 14:19:...|
|        003|          102|    Bob Brown| 35|  Male| 55000|2014-05-01|         M|    Bob Brown|  2026-03-19|2026-03-19 14:19:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|  2026-03-19|2026-03-19 14:19:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Jack 

In [117]:
from pyspark.sql.functions import coalesce,lit

emp_null_df=emp_dated.withColumn("New_gender",coalesce(col("new_gender"),lit("o")))

In [119]:
emp_final=emp_null_df.drop("name","gender").withColumnRenamed("New_name","Name").withColumnRenamed("New_gender","Gender")

In [120]:
emp_final.show()

+-----------+-------------+---+------+----------+------+-------------+------------+--------------------+
|employee_id|department_id|age|salary| hire_date|Gender|         Name|current_date|          Time_stamp|
+-----------+-------------+---+------+----------+------+-------------+------------+--------------------+
|        001|          101| 30| 50000|2015-01-01|     M|     John Doe|  2026-03-19|2026-03-19 14:30:...|
|        002|          101| 25| 45000|2016-02-15|     F|   Jane Smith|  2026-03-19|2026-03-19 14:30:...|
|        003|          102| 35| 55000|2014-05-01|     M|    Bob Brown|  2026-03-19|2026-03-19 14:30:...|
|        004|          102| 28| 48000|2017-09-30|     F|    Alice Lee|  2026-03-19|2026-03-19 14:30:...|
|        005|          103| 40| 60000|2013-04-01|     M|    Jack Chan|  2026-03-19|2026-03-19 14:30:...|
|        006|          103| 32| 52000|2018-07-01|     F|    Jill Wong|  2026-03-19|2026-03-19 14:30:...|
|        007|          101| 42| 70000|2012-03-15|     M

In [123]:
df_reordered=emp_final.select(expr("employee_id as emp_id"),"department_id","Name","age","salary","hire_date","gender","Current_date","Time_stamp")

In [124]:
df_reordered.show()

+------+-------------+-------------+---+------+----------+------+------------+--------------------+
|emp_id|department_id|         Name|age|salary| hire_date|gender|Current_date|          Time_stamp|
+------+-------------+-------------+---+------+----------+------+------------+--------------------+
|   001|          101|     John Doe| 30| 50000|2015-01-01|     M|  2026-03-19|2026-03-19 14:36:...|
|   002|          101|   Jane Smith| 25| 45000|2016-02-15|     F|  2026-03-19|2026-03-19 14:36:...|
|   003|          102|    Bob Brown| 35| 55000|2014-05-01|     M|  2026-03-19|2026-03-19 14:36:...|
|   004|          102|    Alice Lee| 28| 48000|2017-09-30|     F|  2026-03-19|2026-03-19 14:36:...|
|   005|          103|    Jack Chan| 40| 60000|2013-04-01|     M|  2026-03-19|2026-03-19 14:36:...|
|   006|          103|    Jill Wong| 32| 52000|2018-07-01|     F|  2026-03-19|2026-03-19 14:36:...|
|   007|          101|James Johnson| 42| 70000|2012-03-15|     M|  2026-03-19|2026-03-19 14:36:...|


In [125]:
df_reordered.write.format("csv").save("data/output/4/emp.csv")

In [140]:
# convert date into string and print only date
from pyspark.sql.functions import date_format
df_final=df_reordered.withColumn("date_year",to_date(col("time_stamp"),"dd"))

In [141]:
df_final.show()

+------+-------------+-------------+---+------+----------+------+------------+--------------------+----------+
|emp_id|department_id|         Name|age|salary| hire_date|gender|Current_date|          Time_stamp| date_year|
+------+-------------+-------------+---+------+----------+------+------------+--------------------+----------+
|   001|          101|     John Doe| 30| 50000|2015-01-01|     M|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   002|          101|   Jane Smith| 25| 45000|2016-02-15|     F|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   003|          102|    Bob Brown| 35| 55000|2014-05-01|     M|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   004|          102|    Alice Lee| 28| 48000|2017-09-30|     F|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   005|          103|    Jack Chan| 40| 60000|2013-04-01|     M|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|   006|          103|    Jill Wong| 32| 52000|2018-07-01|     F|  2026-03-19|2026-03-19 14:48:...|2026-03-19|
|